# COMP 440 HW2: Whose Preferences Count?

This is the Colab notebook for HW2. Claude Code loads it into Colab from `colab/hw2.ipynb` in
your repository, so everyone starts from the same notebook. Changes made here are not saved back
to your repository, and that's fine: if Colab disconnects, Claude loads the notebook again. You
won't type much here. Claude
Code, on your laptop, adds, changes and runs the cells for you through the Colab MCP server, and
I've put a short note above each cell so you can follow what it does. The notebook needs a T4 GPU.

## Setup

Run these three cells in order whenever you open the notebook or reconnect. A reconnect starts
over on a new machine, so anything loaded before is gone.

### Check the GPU

This cell checks that the notebook has a GPU and prints which one, with the versions of PyTorch
and transformers. If it says there is no GPU, choose Runtime > Change runtime type > T4 GPU and
run it again.

In [ ]:
import torch
import transformers

print(f"torch {torch.__version__}, transformers {transformers.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook has no GPU. In Colab's menu, choose Runtime > Change runtime type > "
        "T4 GPU, then run this cell again.")
gpu = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu}, {memory_gb:.1f} GB of memory")
if "T4" not in gpu:
    print("This GPU is not a T4. The notebook should still work, but times and memory use "
          "will differ from the instructions.")

### Get the test items

This cell downloads the three shared test sets, 240 items in all, from the class repository into
`/content/hw2/items/`. It skips any file that is already there.

In [ ]:
import json
import os
import urllib.request

ITEMS_URL = "https://raw.githubusercontent.com/shilad/comp440-hw2/main/evals/shared"
WORK_DIR = "/content/hw2"
ITEMS_DIR = f"{WORK_DIR}/items"
FAMILIES = ["facts", "user_says_something_wrong", "emotional_social"]

os.makedirs(ITEMS_DIR, exist_ok=True)
items = {}
for family in FAMILIES:
    path = f"{ITEMS_DIR}/{family}.jsonl"
    if os.path.exists(path):
        print(f"{family}.jsonl is already in {ITEMS_DIR}, so it was not downloaded")
    else:
        url = f"{ITEMS_URL}/{family}.jsonl"
        try:
            urllib.request.urlretrieve(url, path + ".part")
        except OSError as e:
            raise RuntimeError(f"Could not download {url} ({e}). Run this cell again in a "
                               "minute. If it keeps failing, ask in #comp440-f26 on Slack.") from e
        os.replace(path + ".part", path)
    with open(path, encoding="utf-8") as f:
        items[family] = [json.loads(line) for line in f if line.strip()]

for family in FAMILIES:
    print(f"{family}: {len(items[family])} items")
print(f"total: {sum(len(v) for v in items.values())} items")

### Load the model

This cell downloads Qwen3-0.6B-Base from Hugging Face and puts it on the GPU, stored as 16-bit
numbers (float16). It takes about 20 seconds, and a warning that you have no Hugging Face token
is fine.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

# transformers 4.56 renamed torch_dtype to dtype. Use the name this version knows.
major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
dtype_arg = "dtype" if (major, minor) >= (4, 56) else "torch_dtype"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **{dtype_arg: torch.float16}).to("cuda")
print(f"Loaded {MODEL_NAME}. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## Part 1

These cells go with Steps 2 and 5 of `instructions/part1.md`.

### Step 2: See what the model predicts next

`top_next_tokens` prints the five most likely next tokens after some text, with their
probabilities. To try another prompt, change the text in the last line and run the cell again.

In [ ]:
def top_next_tokens(text, k=5):
    """Print the k most likely next tokens after text, with their probabilities."""
    ids = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        scores = model(**ids).logits[0, -1]
    probs = torch.softmax(scores.float(), dim=-1)
    top = torch.topk(probs, k)
    for p, i in zip(top.values, top.indices):
        print(f"{p.item():.3f}  {tok.decode(int(i))!r}")

top_next_tokens("Question: Which city is the capital of Canada?\nAnswer:")

### Step 5: Code for answering the test items

This cell only defines functions: `answer_items` gives every test item to a model and records
its answers, and `save_answers` zips them and downloads the zip. Part 2 uses both again for your
trained models.

In [ ]:
import time
import zipfile
from google.colab import files

PROMPT = "Question: {prompt}\nAnswer:"
MAX_NEW_TOKENS = {"facts": 120, "user_says_something_wrong": 120, "emotional_social": 200}
REPETITION_PENALTY = 1.15
BATCH_SIZE = 32

# A copy of the tokenizer for batches. It pads short prompts on the left, because the model
# writes on the right. It pads with a token the model never writes: padding with the
# end-of-text token would let the repetition penalty push the model away from stopping.
batch_tok = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
batch_tok.pad_token = "<|vision_pad|>"


def generate_batch(model, prompts, max_new_tokens):
    """Answer a list of prompts at once. Returns (raw text, number of new tokens) for each."""
    enc = batch_tok([PROMPT.format(prompt=p) for p in prompts], return_tensors="pt",
                    padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             repetition_penalty=REPETITION_PENALTY,
                             pad_token_id=batch_tok.pad_token_id,
                             eos_token_id=batch_tok.eos_token_id)
    results = []
    for row in out[:, enc["input_ids"].shape[1]:].tolist():
        # Keep the tokens before the end-of-text token, if the model wrote one.
        n = row.index(batch_tok.eos_token_id) if batch_tok.eos_token_id in row else len(row)
        results.append((batch_tok.decode(row[:n], skip_special_tokens=True), n))
    return results


def answer_items(model, model_id, items, batch_size=BATCH_SIZE):
    """Give every test item to the model. Returns one row per item, and notes on how."""
    rows, start, first_size = [], time.time(), batch_size
    for family, todo in items.items():
        max_new = MAX_NEW_TOKENS[family]
        done = 0
        while done < len(todo):
            batch = todo[done:done + batch_size]
            try:
                results = generate_batch(model, [it["prompt"] for it in batch], max_new)
            except torch.cuda.OutOfMemoryError:
                results = None
            if results is None:
                if batch_size == 1:
                    raise RuntimeError("Out of GPU memory even with one item at a time. Choose "
                                       "Runtime > Restart session, then run the cells again.")
                batch_size //= 2
                torch.cuda.empty_cache()
                print(f"Out of GPU memory. Trying again with batches of {batch_size}.")
                continue
            for it, (raw, n) in zip(batch, results):
                rows.append({
                    "id": it["id"],
                    "model": model_id,
                    # The base model often goes on to invent the next question. Cut it there.
                    "response": raw.split("\nQuestion:")[0].strip(),
                    "raw": raw,
                    "new_tokens": n,
                    "max_new_tokens": max_new,
                    "hit_limit": n == max_new,
                })
            done += len(batch)
            print(f"{family}: {done} of {len(todo)} done, {time.time() - start:.0f} s")

    seconds = time.time() - start
    hits = sum(r["hit_limit"] for r in rows)
    print(f"Finished {len(rows)} items in {seconds:.0f} s. {hits} ran to the length limit.")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "a CPU"
    sizes = f"{first_size}" if batch_size == first_size else f"{first_size}, then {batch_size}"
    notes = (f"Made {time.strftime('%Y-%m-%d')} with the COMP 440 HW2 Colab notebook: "
             f"{len(rows)} answers in {seconds:.0f} s on {gpu}, weights in "
             f"{str(model.dtype).replace('torch.', '')}, torch {torch.__version__}, "
             f"transformers {transformers.__version__}. Greedy decoding (do_sample=False) with "
             f"repetition_penalty {REPETITION_PENALTY}, in batches of {sizes}, padded on the left. "
             "The response is the raw text cut where the model starts a new line with "
             "'Question:', with spaces trimmed from both ends.")
    return rows, notes


def save_answers(rows, notes, label, description, charter="none", name=None, stands_for=None):
    """Save the answers in the format of a run folder, zip them and download the zip.

    Part 1 saves one model. Part 2 saves four models in one zip: then rows holds every model's
    rows, and notes, label, description and charter are dicts keyed by model id. stands_for
    says which models stand for the succinct and persona predictions."""
    model_ids = list(dict.fromkeys(r["model"] for r in rows))
    model_id = model_ids[0]
    one = len(model_ids) == 1

    def per_model(value, m):
        return value[m] if isinstance(value, dict) else value

    name = name or f"hw2-{model_id}-answers"
    folder = f"{WORK_DIR}/{name}"
    os.makedirs(f"{folder}/responses", exist_ok=True)
    for m in model_ids:
        with open(f"{folder}/responses/{m}.jsonl", "w", encoding="utf-8") as f:
            for r in rows:
                if r["model"] == m:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
    models = [{"id": m, "label": per_model(label, m), "description": per_model(description, m),
               "charter": per_model(charter, m)} for m in model_ids]
    for entry in models:
        if stands_for and entry["id"] in stands_for:
            entry["stands_for"] = stands_for[entry["id"]]
    with open(f"{folder}/models.json", "w", encoding="utf-8") as f:
        json.dump(models, f, indent=2)
    if isinstance(notes, dict):
        notes = "\n".join(f"{m}: {notes[m]}" for m in model_ids)
    with open(f"{folder}/generation-notes.txt", "w", encoding="utf-8") as f:
        f.write(notes + "\n")
    who = f'model "{model_id}"' if one else "models " + ", ".join(f'"{m}"' for m in model_ids)
    responses = f"responses/{model_id}.jsonl" if one else "responses/*.jsonl"
    item_sets = "evals/shared"
    if any(r["id"].startswith("mine-") for r in rows):
        item_sets += " evals/mine"
    with open(f"{folder}/README.txt", "w", encoding="utf-8") as f:
        f.write(f"""Answers from the COMP 440 HW2 Colab notebook, {who}.

{responses}   one line per test item: id, model, response, raw, new_tokens,
    max_new_tokens, hit_limit
models.json   what the {'model is' if one else 'models are'}, for init-run's --models-json
generation-notes.txt   how the answers were made, for init-run's --generation-notes

To make a run from this folder, run this from the repository's folder, with FOLDER the path
to this folder:

python3 evals/shared/grade_tools.py init-run --root evals/runs --name SLUG \\
    --title "..." --description "..." --items {item_sets} \\
    --responses FOLDER/{responses} --models-json FOLDER/models.json \\
    --generation-notes "$(cat FOLDER/generation-notes.txt)"
""")
    zip_path = f"{WORK_DIR}/{name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for part in [f"responses/{m}.jsonl" for m in model_ids] + [
                "models.json", "generation-notes.txt", "README.txt"]:
            z.write(f"{folder}/{part}", f"{name}/{part}")
    print(f"Saved {len(rows)} answers in {zip_path}. Downloading it now.")
    files.download(zip_path)

### Step 5: Answer every test item with the base model

This gives all 240 items to the base model in batches and prints its progress. It should take
about a minute on a T4.

In [ ]:
base_rows, base_notes = answer_items(model, "base", items)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print("The first answer:")
print(json.dumps(base_rows[0], ensure_ascii=False, indent=1))

### Step 5: Save and download the answers

This saves the answers as `responses/base.jsonl`, zips them into `hw2-base-answers.zip` and
downloads the zip to your laptop. The first time, Chrome asks near the address bar whether Colab
may download files; allow it.

In [ ]:
save_answers(base_rows, base_notes, label="Base",
             description="Qwen3-0.6B-Base, the pretrained model with no further training.")

### Free the GPU memory

This takes the model off the GPU, so the memory is free for training in Part 2. To use the model
again, run the Load the model cell again.

In [ ]:
import gc

model = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.2f} GB")

## Part 2

These cells are for Steps 7, 8 and 9 of `instructions/part2.md`.

### Part 2 setup

Part 2 builds on Part 1's cells. Each time you open the notebook or reconnect, run the three
Setup cells and the Step 5 code cell, and then this one. This cell:

- installs TRL, the Hugging Face library that does the training;
- takes the base model off the GPU, because training needs nearly all of the T4's memory;
- downloads your fork from GitHub into `/content/fork`: the example answers, your two pair
  files, `constitutions.md` and your own test items in `evals/mine/`.

Claude puts your fork's address on the `FORK_URL` line. The cell sees only what is on GitHub, so
Claude pushes your commits before running it. It takes about 30 seconds.

In [ ]:
import gc
import glob
import re
import shutil
import subprocess
import sys

# Claude puts your fork's address on this line: your repository's address on GitHub.
FORK_URL = "https://github.com/YOUR-GITHUB-NAME/comp440-hw2"

# The settings below were tested on a free T4. Leave them as they are, unless the out of memory
# note at the end of Part 2 tells you to change one.
BASE_MODEL = "Qwen/Qwen3-0.6B-Base"
FORK_DIR = "/content/fork"
MODELS_DIR = "/content/models"
SFT_DATA = f"{FORK_DIR}/training/sft-dolly-600.jsonl"
READOUT_DATA = f"{FORK_DIR}/training/readout-10.jsonl"
PAIRS_DATA = {"succinct": f"{FORK_DIR}/training/pairs-succinct.jsonl",
              "persona": f"{FORK_DIR}/training/pairs-persona.jsonl"}
OWN_ITEMS_DIR = f"{FORK_DIR}/evals/mine"
SFT_EPOCHS = 2        # passes over the example answers
SFT_BATCH = 8         # answers per step: SFT_BATCH x SFT_ACCUMULATE
SFT_ACCUMULATE = 1
DPO_STEPS = 50        # training steps for each of your two models
DPO_BATCH = 2         # pairs per step: DPO_BATCH x DPO_ACCUMULATE
DPO_ACCUMULATE = 4
TRL_VERSION = "1.14.1"

MODEL_NAMES = ["base", "sft", "succinct", "persona"]
LABELS = {"base": "Base", "sft": "SFT", "succinct": "Succinct", "persona": "Persona"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Part 2 uses Part 1's cells: the shared test items, and the code that answers them.
if "items" not in globals() or "answer_items" not in globals():
    raise RuntimeError("Run the three Setup cells and the Step 5 code cell first, then this cell.")

# 1. Install TRL, the training library. Colab already has PyTorch and transformers.
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"trl=={TRL_VERSION}"],
                     capture_output=True, text=True)
if pip.returncode != 0:
    raise RuntimeError(f"Could not install TRL. Run this cell again in a minute.\n{pip.stderr}")
print(f"TRL {TRL_VERSION} is installed.")

# 2. Take the base model off the GPU.
model = None
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free for training: {free / 2**30:.1f} of {total / 2**30:.1f} GB")

# 3. Download your fork. A fork of the class repository is public, so this needs no password,
# but it gets only what you have pushed.
fork_url = FORK_URL.strip()
if "YOUR-GITHUB-NAME" in fork_url:
    raise RuntimeError("FORK_URL is still the example address. Claude: put the address of the "
                       "student's fork on the FORK_URL line, then run this cell again.")
if fork_url.startswith("git@github.com:"):
    fork_url = "https://github.com/" + fork_url[len("git@github.com:"):]
if re.search(r"github\.com/shilad/comp440-hw2(\.git)?/?$", fork_url, flags=re.I):
    raise RuntimeError("FORK_URL is the class repository. It must be your fork, the copy under "
                       "your own GitHub name.")
shutil.rmtree(FORK_DIR, ignore_errors=True)
clone = subprocess.run(["git", "clone", "--quiet", fork_url, FORK_DIR], capture_output=True,
                       text=True, env=dict(os.environ, GIT_TERMINAL_PROMPT="0"))
if clone.returncode != 0:
    raise RuntimeError(
        f"Could not download your fork from {fork_url}. Check that FORK_URL is your fork's "
        "address and that the fork is public: open the address in a private browser window. If "
        "GitHub can't find it there, the address is wrong or the repository is private.\n"
        f"git said: {clone.stderr.strip()}")


def git(*args):
    return subprocess.run(["git", "-C", FORK_DIR, *args], capture_output=True,
                          text=True).stdout.strip()


FORK_COMMIT = git("log", "-1", "--format=%h")
print(f"Downloaded your fork at commit {git('log', '-1', '--format=%h %ci %s')}")
if "Predictions locked" not in git("log", "--format=%s").splitlines():
    print('WARNING: your fork has no commit named "Predictions locked". Step 4 asks you to '
          "commit your predictions before any training. Don't train until it is pushed.")


# 4. Check the files that training and answering need.
def read_jsonl(path):
    if not os.path.exists(path):
        raise RuntimeError(f"{os.path.relpath(path, FORK_DIR)} is not in your fork on GitHub. "
                           "If it is on your laptop, Claude: commit and push it, then run this "
                           "cell again.")
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


sft_count = len(read_jsonl(SFT_DATA))
readout = read_jsonl(READOUT_DATA)
pair_counts = {name: len(read_jsonl(path)) for name, path in PAIRS_DATA.items()}
print(f"{sft_count} example answers for SFT; {pair_counts['succinct']} succinct pairs and "
      f"{pair_counts['persona']} persona pairs for DPO; {len(readout)} readout questions")

own_items = {}
own_files = sorted(glob.glob(f"{OWN_ITEMS_DIR}/*.jsonl"))
if not own_files:
    raise RuntimeError("Your fork has no test items in evals/mine/. If your two families are on "
                       "your laptop, Claude: commit and push them, then run this cell again.")
for path in own_files:
    file_name = os.path.basename(path)
    if "hidden" in file_name:
        raise RuntimeError(f'evals/mine/{file_name}: "hidden" in a file name marks held-back '
                           "items. Rename the file, push, and run this cell again.")
    for it in read_jsonl(path):
        if not (it.get("id") and it.get("prompt") and str(it.get("family")).startswith("mine-")
                and it.get("graded_like") in FAMILIES):
            raise RuntimeError(
                f"evals/mine/{file_name}: item {it.get('id')} needs an id, a prompt, a family "
                'that starts with "mine-" and graded_like set to one of ' + ", ".join(FAMILIES)
                + ". Claude: check the file with validate.py, push the fix, and run this cell "
                "again.")
        own_items.setdefault(it["family"], []).append(it)
for family, todo in own_items.items():
    # Your items get the answer length of the shared family they are graded like.
    MAX_NEW_TOKENS[family] = max(MAX_NEW_TOKENS[it["graded_like"]] for it in todo)
test_items = {**items, **own_items}
print("Your own items: " + ", ".join(f"{f} {len(v)}" for f, v in own_items.items()))
print(f"Step 9 will give {sum(len(v) for v in test_items.values())} items to each model.")


def constitution_style(heading):
    """The text under '## <heading>' in your constitutions.md, without comments ('' if none)."""
    path = f"{FORK_DIR}/constitutions.md"
    if not os.path.exists(path):
        return ""
    with open(path, encoding="utf-8") as f:
        text = re.sub(r"<!--.*?-->", "", f.read(), flags=re.S)
    found = re.search(rf"^## {re.escape(heading)}[ \t]*\n(.*?)(?=^## |\Z)", text,
                      flags=re.M | re.S)
    return found.group(1).strip() if found else ""


CHARTERS = {"base": "none", "sft": "none (the Dolly answers' own style)",
            "succinct": constitution_style("Succinct style"),
            "persona": constitution_style("Persona style")}
for name in ["succinct", "persona"]:
    if not CHARTERS[name] or "XXXX" in CHARTERS[name]:
        raise RuntimeError(f'The "{LABELS[name]} style" section of constitutions.md in your '
                           "fork is not written yet: it still reads XXXX, or is empty. Write it "
                           "(Part 2, Step 2), push, and run this cell again.")


# 5. Code that Steps 7 to 9 use.
def run_script(*args):
    """Run a Python script as its own process and print what it prints. When the process ends,
    all the GPU memory it used is free again, even if it failed."""
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               TOKENIZERS_PARALLELISM="false")
    proc = subprocess.Popen([sys.executable, "-u", *[str(a) for a in args]], env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for line in proc.stdout:
            print(line, end="")
    finally:
        if proc.poll() is None:  # The cell was stopped, so stop the script too.
            proc.kill()
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("The script stopped with an error; its message is above. If it says "
                           "CUDA out of memory, read the last cell of Part 2.")


def model_path(name):
    """The base model's name on Hugging Face, or the folder of a model trained here."""
    if name == "base":
        return BASE_MODEL
    path = f"{MODELS_DIR}/{name}"
    if not os.path.exists(f"{path}/train-log.json"):
        raise RuntimeError(f"There is no trained {LABELS[name]} model in {path}. Run its training "
                           "cell first. After a reconnect the trained models are gone, so run "
                           "the notebook again from the top.")
    return path


def load_model(name):
    """Load a model to answer questions: on the GPU, in 16-bit numbers."""
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    return AutoModelForCausalLM.from_pretrained(model_path(name), **{dtype_arg: dtype}).to(DEVICE)


def free_gpu():
    """Give back the GPU memory that no variable uses any more."""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_readout(names, questions, max_new_tokens=200):
    """Give a few readout questions to each model in turn, then print the answers together."""
    answers = {}
    for name in names:
        m = load_model(name)
        results = generate_batch(m, [q["question"] for q in questions], max_new_tokens)
        answers[name] = [raw.split("\nQuestion:")[0].strip() for raw, _ in results]
        m = None
        free_gpu()
    for i, q in enumerate(questions):
        print(f"=== {q['question']}")
        for name in names:
            print(f"--- {LABELS[name]}:\n{answers[name][i]}")
        print()

### Step 7: The SFT training script

This cell only writes a file, `/content/hw2/train_sft.py`; the next cell runs it. The script
trains every weight of the base model on the 600 example answers in
`training/sft-dolly-600.jsonl`, two passes, 8 answers per step, and saves the trained model in
`/content/models/sft`. It runs as its own Python process. Training uses nearly all of the T4's
memory, and a process gives all of its memory back when it ends, even if it fails.

In [ ]:
%%writefile /content/hw2/train_sft.py
"""Step 7: supervised fine-tuning (SFT). Trains every weight of the base model on example
answers written by people, then saves it in 16-bit numbers. The notebook runs this script as
its own process, so all the GPU memory it uses is free again when it ends."""
import argparse
import json
import math
import time

import datasets
import torch
import transformers
import trl
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from transformers.trainer_callback import PrinterCallback
from trl import SFTConfig, SFTTrainer

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True, help="the base model's name on Hugging Face")
parser.add_argument("--data", required=True, help="the example answers, one JSON line each")
parser.add_argument("--out", required=True, help="the folder to save the trained model in")
parser.add_argument("--epochs", type=float, default=2, help="passes over the examples")
parser.add_argument("--batch", type=int, default=8, help="examples the GPU takes at once")
parser.add_argument("--accumulate", type=int, default=1, help="batches per training step")
parser.add_argument("--learning-rate", type=float, default=2e-5)
parser.add_argument("--max-length", type=int, default=384, help="longest example, in tokens")
parser.add_argument("--fork-commit", default="", help="the fork commit the data came from")
args = parser.parse_args()

start = time.time()
gpu = torch.cuda.is_available()
torch.manual_seed(0)
datasets.disable_progress_bars()  # Progress bars print badly from a separate process.
transformers.utils.logging.disable_progress_bar()
if gpu:
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free at the start: {free / 2**30:.1f} of {total / 2**30:.1f} GB")

with open(args.data, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
# Each example is a prompt ("Question: ...\nAnswer:") and a completion, the answer. The loss
# counts only the completion, so the model learns to write answers, not questions.
data = Dataset.from_list([{"prompt": r["prompt"], "completion": r["completion"]} for r in rows])

tok = AutoTokenizer.from_pretrained(args.model)
# The weights stay in 32-bit numbers. fp16 below does most of the arithmetic in 16-bit numbers,
# which the T4 does about six times faster.
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float32)
too_long = sum(len(tok(r["prompt"] + r["completion"]).input_ids) >= args.max_length for r in rows)
if too_long:
    print(f"{too_long} of {len(rows)} examples are longer than {args.max_length} tokens; training "
          "uses only their beginning.")

steps = math.ceil(math.ceil(len(rows) / (args.batch * args.accumulate)) * args.epochs)
config = SFTConfig(
    output_dir=args.out, num_train_epochs=args.epochs, per_device_train_batch_size=args.batch,
    gradient_accumulation_steps=args.accumulate, learning_rate=args.learning_rate,
    lr_scheduler_type="cosine", warmup_steps=5, weight_decay=0.0, max_length=args.max_length,
    fp16=gpu, bf16=False, use_cpu=not gpu,  # The T4 has no fast bfloat16, another 16-bit format.
    logging_steps=max(1, steps // 6),  # about six loss lines: every 25 steps of 150
    save_strategy="no", report_to="none", disable_tqdm=True, dataloader_num_workers=0, seed=0)
loss_log = []


class PrintLoss(TrainerCallback):
    """Print the average loss since the last line, with the minutes since training began."""

    def on_log(self, _args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            loss_log.append({"step": state.global_step, "loss": round(logs["loss"], 4)})
            print(f"step {state.global_step} of {state.max_steps}: loss {logs['loss']:.2f} "
                  f"({(time.time() - train_start) / 60:.1f} min)", flush=True)


trainer = SFTTrainer(model=model, args=config, train_dataset=data, processing_class=tok,
                     callbacks=[PrintLoss()])
trainer.remove_callback(PrinterCallback)  # It would print every log a second time.
print(f"Training on {len(rows)} example answers: {args.epochs:g} passes, "
      f"{args.batch * args.accumulate} answers per step, {steps} steps.", flush=True)
train_start = time.time()
result = trainer.train()
minutes = (time.time() - train_start) / 60

# Save in 16-bit numbers: half the size, and the notebook answers in 16-bit anyway.
trainer.model.config.use_cache = True
trainer.model.to(torch.float16).save_pretrained(args.out)
tok.save_pretrained(args.out)
log = {
    "stage": "sft", "model": args.model, "data": args.data, "fork_commit": args.fork_commit,
    "examples": len(rows), "epochs": args.epochs, "batch": args.batch,
    "accumulate": args.accumulate, "learning_rate": args.learning_rate,
    "max_length": args.max_length, "steps": trainer.state.global_step, "loss_log": loss_log,
    "final_loss": round(result.training_loss, 4), "train_minutes": round(minutes, 2),
    "total_minutes": round((time.time() - start) / 60, 2),
    "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if gpu else None,
    "peak_gpu_reserved_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2) if gpu else None,
    "device": torch.cuda.get_device_name(0) if gpu else "CPU",
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "trl": trl.__version__}}
with open(f"{args.out}/train-log.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=1)
memory = f", peak GPU memory {log['peak_gpu_gb']} GB" if gpu else ""
print(f"Done: {log['steps']} steps in {minutes:.1f} minutes, average loss "
      f"{result.training_loss:.2f}{memory}. Saved in {args.out}")

### Step 7: Train on the example answers (SFT)

This runs the script. It prints the loss every 25 steps, 150 steps in all, and then how many
minutes training took. Expect about 3 minutes on a T4. Keep your laptop awake until it finishes:
if Colab disconnects, the trained model is lost and Part 2 starts over.

In [ ]:
shutil.rmtree(f"{MODELS_DIR}/sft", ignore_errors=True)
run_script(f"{WORK_DIR}/train_sft.py", "--model", BASE_MODEL, "--data", SFT_DATA,
           "--out", f"{MODELS_DIR}/sft", "--epochs", SFT_EPOCHS, "--batch", SFT_BATCH,
           "--accumulate", SFT_ACCUMULATE, "--fork-commit", FORK_COMMIT)

### Step 7: Compare the base and SFT answers

This gives the first five questions in `training/readout-10.jsonl` to the base model and then to
your SFT model, and prints both answers under each question. Neither model trained on these
questions. It takes about a minute.

In [ ]:
show_readout(["base", "sft"], readout[:5])

### Step 8: The DPO training script

This cell writes `/content/hw2/train_dpo.py`; the next two cells run it, once for each of your
models. The script starts from your SFT model and trains every weight on one of your pair files.
For each pair it makes the chosen answer more likely and the rejected answer less likely,
measured against a frozen copy of the SFT model. It runs 50 steps of 8 pairs and saves the
model. Like the SFT script, it runs as its own process.

In [ ]:
%%writefile /content/hw2/train_dpo.py
"""Step 8: preference training (DPO). Starts from the SFT model and trains every weight on pairs
of answers: for each prompt it makes the chosen answer more likely and the rejected one less
likely, compared with a frozen copy of the SFT model. Then it saves the model in 16-bit numbers.
The notebook runs this script as its own process, so its GPU memory is freed when it ends."""
import argparse
import json
import os
import time

import datasets
import torch
import transformers
import trl
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from transformers.trainer_callback import PrinterCallback
from trl import DPOConfig, DPOTrainer

parser = argparse.ArgumentParser()
parser.add_argument("--model", required=True, help="the SFT model's folder")
parser.add_argument("--pairs", required=True, help="the pairs, one JSON line each")
parser.add_argument("--out", required=True, help="the folder to save the trained model in")
parser.add_argument("--steps", type=int, default=50, help="training steps")
parser.add_argument("--batch", type=int, default=2, help="pairs the GPU takes at once")
parser.add_argument("--accumulate", type=int, default=4, help="batches per training step")
parser.add_argument("--learning-rate", type=float, default=5e-6)
parser.add_argument("--beta", type=float, default=0.1, help="how far from the SFT model to go")
parser.add_argument("--max-length", type=int, default=320, help="longest prompt and answer")
parser.add_argument("--fork-commit", default="", help="the fork commit the pairs came from")
args = parser.parse_args()

start = time.time()
gpu = torch.cuda.is_available()
torch.manual_seed(0)
datasets.disable_progress_bars()  # Progress bars print badly from a separate process.
transformers.utils.logging.disable_progress_bar()
if gpu:
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free at the start: {free / 2**30:.1f} of {total / 2**30:.1f} GB")

with open(args.pairs, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
data = Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"],
                           "rejected": r["rejected"]} for r in rows])

tok = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float32)
# The frozen copy is only read, never trained, so 16-bit numbers are enough. That saves memory.
reference = AutoModelForCausalLM.from_pretrained(
    args.model, dtype=torch.float16 if gpu else torch.float32)
too_long = sum(max(len(tok(r["prompt"] + r[side]).input_ids) for side in ("chosen", "rejected"))
               >= args.max_length for r in rows)
if too_long:
    print(f"{too_long} of {len(rows)} pairs have an answer longer than {args.max_length} tokens "
          "with its prompt; training uses only the beginning of that answer.")

config = DPOConfig(
    output_dir=args.out, max_steps=args.steps, per_device_train_batch_size=args.batch,
    gradient_accumulation_steps=args.accumulate, learning_rate=args.learning_rate,
    lr_scheduler_type="cosine", warmup_steps=3, beta=args.beta, max_length=args.max_length,
    fp16=gpu, bf16=False, use_cpu=not gpu,  # The T4 has no fast bfloat16, another 16-bit format.
    # Recompute some numbers in the backward pass instead of keeping them: slower, but DPO
    # doesn't fit in the T4's memory without it.
    gradient_checkpointing=True,
    logging_steps=max(1, args.steps // 10),  # every 5 steps of 50
    save_strategy="no", report_to="none", disable_tqdm=True, dataloader_num_workers=0, seed=0)
reward_log = []


class PrintRewards(TrainerCallback):
    """Print the loss and the reward log, averaged over the steps since the last line."""

    def on_log(self, _args, state, control, logs=None, **kwargs):
        if logs and "rewards/chosen" in logs:
            row = {"step": state.global_step, "loss": round(logs["loss"], 4),
                   "chosen": round(logs["rewards/chosen"], 3),
                   "rejected": round(logs["rewards/rejected"], 3),
                   "accuracy": round(logs["rewards/accuracies"], 3)}
            reward_log.append(row)
            print(f"step {row['step']} of {state.max_steps}: loss {logs['loss']:.3f}, chosen "
                  f"reward {row['chosen']:+.2f}, rejected reward {row['rejected']:+.2f}, "
                  f"accuracy {row['accuracy']:.2f} ({(time.time() - train_start) / 60:.1f} min)",
                  flush=True)


trainer = DPOTrainer(model=model, ref_model=reference, args=config, train_dataset=data,
                     processing_class=tok, callbacks=[PrintRewards()])
trainer.remove_callback(PrinterCallback)  # It would print every log a second time.
print(f"Training on {len(rows)} pairs from {os.path.basename(args.pairs)}: "
      f"{args.steps} steps of {args.batch * args.accumulate} pairs.", flush=True)
train_start = time.time()
result = trainer.train()
minutes = (time.time() - train_start) / 60

# Save in 16-bit numbers. Training turned off the cache that makes answering fast; turn it on.
trainer.model.config.use_cache = True
trainer.model.to(torch.float16).save_pretrained(args.out)
tok.save_pretrained(args.out)
log = {
    "stage": "dpo", "model": args.model, "pairs_file": args.pairs, "fork_commit": args.fork_commit,
    "pairs": len(rows), "steps": trainer.state.global_step, "batch": args.batch,
    "accumulate": args.accumulate, "learning_rate": args.learning_rate, "beta": args.beta,
    "max_length": args.max_length, "reward_log": reward_log,
    "final_loss": round(result.training_loss, 4), "train_minutes": round(minutes, 2),
    "total_minutes": round((time.time() - start) / 60, 2),
    "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if gpu else None,
    "peak_gpu_reserved_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2) if gpu else None,
    "device": torch.cuda.get_device_name(0) if gpu else "CPU",
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "trl": trl.__version__}}
with open(f"{args.out}/train-log.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=1)
memory = f", peak GPU memory {log['peak_gpu_gb']} GB" if gpu else ""
print(f"Done: {log['steps']} steps in {minutes:.1f} minutes{memory}. Saved in {args.out}")

# The reward log at steps 5, 25 and 50, or the nearest steps that have a line.
shown = []
for target in (5, 25, trainer.state.global_step):
    if reward_log:
        row = min(reward_log, key=lambda r: abs(r["step"] - target))
        if row not in shown:
            shown.append(row)
print("\nstep   chosen reward   rejected reward   accuracy")
for row in shown:
    print(f"{row['step']:>4}   {row['chosen']:>+13.2f}   {row['rejected']:>+15.2f}   "
          f"{row['accuracy']:>8.2f}")

### Step 8: Train the Succinct model

This trains on `training/pairs-succinct.jsonl` and saves the model in `/content/models/succinct`.
Every 5 steps it prints the loss and the reward log: the average reward of the chosen answers and
of the rejected answers, and the accuracy, the share of pairs whose chosen answer got the higher
reward. At the end it prints the log at steps 5, 25 and 50 again. Expect about 3 minutes.

In [ ]:
shutil.rmtree(f"{MODELS_DIR}/succinct", ignore_errors=True)
run_script(f"{WORK_DIR}/train_dpo.py", "--model", model_path("sft"),
           "--pairs", PAIRS_DATA["succinct"], "--out", f"{MODELS_DIR}/succinct",
           "--steps", DPO_STEPS, "--batch", DPO_BATCH, "--accumulate", DPO_ACCUMULATE,
           "--fork-commit", FORK_COMMIT)

### Step 8: Train the Persona model

The same as the cell above, with `training/pairs-persona.jsonl`. It saves the model in
`/content/models/persona`. Expect about 3 minutes.

In [ ]:
shutil.rmtree(f"{MODELS_DIR}/persona", ignore_errors=True)
run_script(f"{WORK_DIR}/train_dpo.py", "--model", model_path("sft"),
           "--pairs", PAIRS_DATA["persona"], "--out", f"{MODELS_DIR}/persona",
           "--steps", DPO_STEPS, "--batch", DPO_BATCH, "--accumulate", DPO_ACCUMULATE,
           "--fork-commit", FORK_COMMIT)

### Step 8: Compare the answers

This gives the last five readout questions to your SFT, Succinct and Persona models and prints
the three answers under each question. It takes about a minute.

In [ ]:
show_readout(["sft", "succinct", "persona"], readout[5:10])

### Step 9: Answer every test item with all four models

This gives the 240 shared items and your own items to the Base, SFT, Succinct and Persona models,
one model at a time, with the same settings as Part 1, and prints its progress. It takes about 6
to 7 minutes.

In [ ]:
for name in MODEL_NAMES:
    model_path(name)  # Stops here, before any answering, if a trained model is missing.
four_rows, four_notes = [], {}
for name in MODEL_NAMES:
    print(f"Answering with the {LABELS[name]} model.")
    m = load_model(name)
    rows, four_notes[name] = answer_items(m, name, test_items)
    four_rows += rows
    m = None
    free_gpu()
print(f"Done: {len(four_rows)} answers from {len(MODEL_NAMES)} models.")

### Step 9: Save and download the answers

This saves all four models' answers in one zip, `hw2-four-models.zip`, and downloads it to your
laptop. It is laid out like the Part 1 zip, with one answers file per model, and its
`models.json` describes each model. If the download doesn't start, run this cell again.

In [ ]:
logs = {}
for name in ["sft", "succinct", "persona"]:
    with open(f"{model_path(name)}/train-log.json", encoding="utf-8") as f:
        logs[name] = json.load(f)
sft = logs["sft"]
descriptions = {
    "base": f"{BASE_MODEL.split('/')[-1]}, the pretrained model with no further training.",
    "sft": (f"The base model after supervised fine-tuning (SFT): {sft['epochs']:g} passes over "
            f"{sft['examples']} Dolly questions with answers written by people, all weights "
            f"trained ({sft['steps']} steps of {sft['batch'] * sft['accumulate']} answers, "
            f"learning rate {sft['learning_rate']:g}). Trained from commit "
            f"{sft['fork_commit']} of the fork."),
}
for name in ["succinct", "persona"]:
    d = logs[name]
    descriptions[name] = (
        f"The SFT model after DPO preference training on {d['pairs']} pairs from "
        f"training/pairs-{name}.jsonl, in which Claude's answer written under the "
        f"{LABELS[name]} style of constitutions.md is chosen over the human Dolly answer. "
        f"{d['steps']} steps of {d['batch'] * d['accumulate']} pairs, learning rate "
        f"{d['learning_rate']:g}, beta {d['beta']:g}, all weights trained. Trained from commit "
        f"{d['fork_commit']} of the fork.")
save_answers(four_rows, four_notes, label=LABELS, description=descriptions, charter=CHARTERS,
             name="hw2-four-models", stands_for={"succinct": "succinct", "persona": "persona"})

### If you see CUDA out of memory

A training cell that runs out of GPU memory gives the memory back when its process stops, so
first run that cell again. If it runs out again, or a cell that answers questions runs out of
memory, run the cell below to free the memory the notebook itself is holding. It restarts only
Python: the T4, the files in `/content` and your trained models in `/content/models` all stay.
Colab will say the session crashed; that's expected. Then run the three Setup cells, the Step 5
code cell and the Part 2 setup cell, and go on from the step that failed.

If SFT still runs out of memory after that, change `SFT_BATCH = 8` to `SFT_BATCH = 4` and
`SFT_ACCUMULATE = 1` to `SFT_ACCUMULATE = 2` in the Part 2 setup cell, then run that cell and the
SFT cell again. The model still learns from 8 answers per step; it takes them 4 at a time, which
is a little slower. For DPO, change `DPO_BATCH = 2` to `DPO_BATCH = 1` and `DPO_ACCUMULATE = 4`
to `DPO_ACCUMULATE = 8`.

Claude must not use the Colab server's `restart_runtime` tool for this. It gives the machine back
to Google, and your trained models go with it.

In [ ]:
# Restarts only Python. The files in /content, your trained models among them, stay.
import os

os.kill(os.getpid(), 9)